# Data Quality and Schema Audit

## Business question

Is the complaint dataset reliable enough for decision support, and where are the main data-quality risks?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from cx_complaints.data.schema import CANONICAL_COLUMNS, normalize_complaints

plt.style.use("ggplot")

## 1. Load and normalize data

In [ ]:
raw_path = Path("data/raw/cfpb_sample.csv")
if not raw_path.exists():
    raise FileNotFoundError(
        "Expected data/raw/cfpb_sample.csv. Run the fetch command first."
    )

raw = pd.read_csv(raw_path)
complaints = normalize_complaints(raw)

print(f"Raw rows: {len(raw):,}")
print(f"Normalized rows: {len(complaints):,}")
complaints.shape

## 2. Schema conformance checks

In [ ]:
schema_check = pd.DataFrame(
    {
        "column": list(CANONICAL_COLUMNS),
        "present": [column in complaints.columns for column in CANONICAL_COLUMNS],
        "dtype": [
            str(complaints[column].dtype) if column in complaints.columns else "missing"
            for column in CANONICAL_COLUMNS
        ],
    }
)
schema_check

In [ ]:
duplicate_ids = complaints["complaint_id"].duplicated(keep=False).sum()
invalid_dates = complaints["date_received"].isna().sum()

integrity_summary = pd.DataFrame(
    {
        "check": [
            "Duplicate complaint_id rows",
            "Missing/invalid date_received rows",
            "Missing complaint_id rows",
        ],
        "count": [
            int(duplicate_ids),
            int(invalid_dates),
            int(complaints["complaint_id"].isna().sum()),
        ],
    }
)
integrity_summary

## 3. Missingness profile

In [ ]:
missingness = pd.DataFrame(
    {
        "column": complaints.columns,
        "missing_count": complaints.isna().sum().values,
    }
)
missingness["missing_pct"] = (missingness["missing_count"] / len(complaints) * 100).round(2)
missingness = missingness.sort_values("missing_pct", ascending=False)
missingness.head(20)

In [ ]:
top_missing = missingness.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_missing["column"], top_missing["missing_pct"], color="#B56576")
ax.set_title("Top Missing Fields")
ax.set_xlabel("Missing share (%)")
fig.tight_layout()

## 4. Narrative availability over time

In [ ]:
complaints["date_received"] = pd.to_datetime(complaints["date_received"], errors="coerce")
with_dates = complaints.dropna(subset=["date_received"]).copy()
with_dates["month"] = with_dates["date_received"].dt.to_period("M").astype(str)

narrative_monthly = (
    with_dates.groupby("month")
    .agg(
        complaints=("complaint_id", "size"),
        narrative_share_pct=("has_narrative", lambda s: s.fillna(False).astype(bool).mean() * 100),
    )
    .reset_index()
    .sort_values("month")
)
narrative_monthly["narrative_share_pct"] = narrative_monthly["narrative_share_pct"].round(2)
narrative_monthly.tail(12)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(
    narrative_monthly["month"],
    narrative_monthly["complaints"],
    marker="o",
    label="Complaints",
)
ax2.plot(
    narrative_monthly["month"],
    narrative_monthly["narrative_share_pct"],
    marker="s",
    color="#2A9D8F",
    label="Narrative share (%)",
)

ax1.set_title("Complaint Volume and Narrative Coverage by Month")
ax1.set_xlabel("Month")
ax1.set_ylabel("Complaint count")
ax2.set_ylabel("Narrative share (%)")
ax1.tick_params(axis="x", rotation=45)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
fig.tight_layout()

## 5. QA conclusions

Complete after running cells:

1. Highlight critical missing fields and mitigation plans.
2. Confirm ID/date integrity status.
3. Document whether narrative coverage is stable enough for NLP-heavy outputs.

In [ ]:
# Optional export for monitoring reports
output_dir = Path("reports/notebook")
output_dir.mkdir(parents=True, exist_ok=True)

schema_check.to_csv(output_dir / "notebook_schema_check.csv", index=False)
integrity_summary.to_csv(output_dir / "notebook_integrity_summary.csv", index=False)
missingness.to_csv(output_dir / "notebook_missingness_profile.csv", index=False)
narrative_monthly.to_csv(output_dir / "notebook_narrative_monthly.csv", index=False)

output_dir